# Model Updates

`model.update(...)` and `model.select(...).update(...)` apply explicit schema mutations to selected nodes. Use this when a base schema is right, but a specific workflow needs different masking, targets, embeddings, or serving behavior.

Common mutations:

| Mutation | Purpose |
| --- | --- |
| `target=True` | Withhold a field and decode it as a supervised target. |
| `p_mask=...` | Randomly hide values for self-supervised reconstruction. |
| `p_prune=...` | Remove whole field instances from input. |
| `embed=True` | Return embeddings from a selected node. |
| `target=False` | Clear target behavior so the field is encoded as an input when present. |

Start with a plain model. The species field is already a target, but the rest of the schema has no masking or embedding behavior yet.

In [1]:
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()

In [2]:
model = j2v.Model.from_schema(
    j2v.Number("sepal_length"),
    j2v.Number("petal_length"),
    j2v.Category("species", target=True, max_vocab_size=4),
    d_model=16,
    n_layers=1,
    n_heads=4,
    embed=True,
)


Predicates are composable. Here, the example selects all numbers, all categories except the target, the target itself, and finally the root record node. Keep broad selectors visible by plotting the model and checking `last_mutation` before trusting them in a larger workflow.

In [3]:
numbers = j2v.where("type") == "number"
categories = j2v.where("type") == "category"
targets = j2v.where("name") == "species"

_ = model.update(numbers, p_mask=0.15)
_ = model.update(categories & ~targets, p_mask=0.05)
_ = model.update(targets, target=True)


Plot after mutation to verify the final schema state. This is usually faster than reading nested configuration objects by hand.

In [4]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  18,153                                │
│  Schema map                                                            arrays  1                                     │
│                                                                        fields  3                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               

Every mutation records what changed. `last_mutation` is useful in notebooks and tests when you want to confirm a broad selector touched only the intended nodes.

In [5]:
pprint(model.hyperparameters.last_mutation)

MutationResult(
│   operation_id='761d706a80774eb6914e5c75119ff613',
│   parent_operation_id=None,
│   action='update',
│   matched=1,
│   updated=0,
│   skipped=0,
│   changes=()
)